In [1]:
#由于cifar10数据集一直下载失败所以此处省略完整训练流程的代码，补在gpu后面了

`output.argmax(1)`行方向求最大值索引

In [2]:
import torch

# 模拟模型的输出结果，形状为 (2, 2)，代表 2 个样本，每个样本预测 2 个类别的得分/概率
outputs = torch.tensor([[0.1, 0.2],  # 样本 1：预测类别 0 的概率是 0.1，类别 1 的概率是 0.2
                        [0.3, 0.4]]) # 样本 2：预测类别 0 的概率是 0.3，类别 1 的概率是 0.4

# argmax(1) 表示沿着 dim=1（横向/行方向）寻找最大值的索引
# 样本 1 中最大值是 0.2，索引是 1；样本 2 中最大值是 0.4，索引是 1
result = outputs.argmax(1)
# 输出结果为 tensor([1, 1])，代表这两个样本模型预测的都是“类别 1”

`output.argmax(0)`列方向求最大值索引

In [3]:
import torch

# 注意：这里第二行的第一个数字改成了 0.03
outputs = torch.tensor([[0.1,  0.2],   # 第一行
                        [0.03, 0.4]])  # 第二行

# argmax(0) 表示沿着 dim=0（竖向/列方向）寻找最大值的索引
# 第一列 [0.1, 0.03] 中最大值是 0.1，在第 0 行，所以索引是 0
# 第二列 [0.2, 0.4]  中最大值是 0.4，在第 1 行，所以索引是 1
result = outputs.argmax(0)

# 输出结果为 tensor([0, 1])


计算二分类任务的正确率中的应用

In [4]:
import torch

# 1. 模拟模型输出（2个样本，2个类别）
outputs = torch.tensor([[0.1, 0.2],   # 样本 1
                        [0.3, 0.4]])  # 样本 2

# 2. 打印横向最大值的索引
print(outputs.argmax(1)) # 结果是 tensor([1, 1])

# 3. 将模型预测的类别索引保存到变量 preds 中
preds = outputs.argmax(1)

# 4. 定义真实的标签（Targets）
targets = torch.tensor([0, 1]) # 真实情况是：样本 1 的标签是 0，样本 2 的标签是 1

# 5. 将预测结果与真实标签进行逐个元素对比（==）
# preds 是 [1, 1]，targets 是 [0, 1]，样本 1：1 == 0 -> False (预测错了)，样本 2：1 == 1 -> True  (预测对了)
print(preds == targets) # 结果是 tensor([False, True])


# 6. 对对比结果进行求和（.sum()）
# 在 PyTorch 中，求和时 True 会被当成 1，False 会被当成 0，0 + 1 = 1，代表这批数据里一共预测对了 1 个样本
print((preds == targets).sum()) # 结果是 tensor(1)

tensor([1, 1])
tensor([False,  True])
tensor(1)


利用GPU训练的两种方法

In [6]:
'''1'''
# 1. 判断 GPU 是否可用
if torch.cuda.is_available():
    model = model.cuda()          # 模型转移（不强制赋值，但规范写也行）
    loss_fn = loss_fn.cuda()      # 损失函数转移（必须赋值）

# 2. 训练循环中
for imgs, labels in trainloader:
    if torch.cuda.is_available():
        imgs = imgs.cuda()        # 输入数据转移（必须赋值）
        labels = labels.cuda()    # 标注数据转移（必须赋值）

NameError: name 'model' is not defined

In [7]:
'''2'''
# 1. 定义设备（有 GPU 用 GPU，没有就用 CPU）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 2. 将模型和损失函数转移到指定设备
model = model.to(device)
loss_fn = loss_fn.to(device)

# 3. 训练循环中转移数据
for imgs, labels in trainloader:
    imgs = imgs.to(device)
    labels = labels.to(device)

NameError: name 'model' is not defined

例：cifar10

In [8]:
#找到变量网络模型，数据(输入，标注），损失函数，调用.cuda，需要判断if torch.cuda.is.available():，且需要返回值，可以尝试记录start time和end time查看对比训练速度

import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import time

# 步骤 1: 定义设备（判断 GPU 是否可用）
# 对应知识点：if torch.cuda.is_available():，这里使用更现代的 torch.device 写法，它能自动根据是否有 GPU 来选择设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前正在使用的训练设备是: {device}")

# 步骤 2: 准备数据（这里用 CIFAR-10 数据集演示）
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 如果已经在 Google Drive 下载了，可以Drive 路径，这里默认下载到 Colab 本地
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

# 步骤 3: 搭建网络模型并转移到 GPU
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 3, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.classifier(x)

# 实例化模型
model = MyModel()

# 核心点 1: 将网络模型转移到 GPU，也可以写成 model = model.cuda()
model = model.to(device)

# 步骤 4: 定义损失函数与优化器并转移到 GPU
# 核心点 2: 将损失函数转移到 GPU，也可以写成 loss_fn = nn.CrossEntropyLoss().cuda()
loss_fn = nn.CrossEntropyLoss().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 步骤 5: 开始训练并记录时间
print("开始训练...")

# 对应知识点：记录开始时间
start_time = time.time()

# 为了演示时间对比，只跑 2 个 Epoch
epochs = 2

for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # 获取输入数据（imgs 是输入，labels 是标注）
        imgs, labels = data

        # 核心点 3: 将数据和标注转移到 GPU，并且必须重新赋值。也可以写成：imgs = imgs.cuda(), labels = labels.cuda()
        imgs = imgs.to(device)
        labels = labels.to(device)

        # 梯度清零
        optimizer.zero_grad()

        # 前向传播
        outputs = model(imgs)

        # 计算损失
        loss = loss_fn(outputs, labels)

        # 反向传播
        loss.backward()

        # 优化权重
        optimizer.step()

        running_loss += loss.item()
        if i % 100 == 99:    # 每 100 个 batch 打印一次进度
            print(f'[Epoch {epoch + 1}, Batch {i + 1}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0

# 对应知识点：记录结束时间并计算差值
end_time = time.time()
total_time = end_time - start_time

print("----------------------------------------")
print(f"训练结束！总耗时: {total_time:.2f} 秒")
print("----------------------------------------")


当前正在使用的训练设备是: cuda


100%|██████████| 170M/170M [41:48<00:00, 68.0kB/s]


开始训练...
[Epoch 1, Batch 100] loss: 1.795
[Epoch 1, Batch 200] loss: 1.642
[Epoch 1, Batch 300] loss: 1.558
[Epoch 2, Batch 100] loss: 1.445
[Epoch 2, Batch 200] loss: 1.425
[Epoch 2, Batch 300] loss: 1.413
----------------------------------------
训练结束！总耗时: 25.10 秒
----------------------------------------


In [ ]:
#找到变量网络模型，数据(输入，标注），损失函数，调用.cuda，需要判断if torch.cuda.is.available():，且需要返回值，可以尝试记录start time和end time查看对比训练速度
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
import time

# 步骤 1: 定义设备（判断 GPU 是否可用）
# 对应知识点：if torch.cuda.is_available():，这里使用更现代的 torch.device 写法，它能自动根据是否有 GPU 来选择设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前正在使用的训练设备是: {device}")

# 步骤 2: 准备数据（这里用 CIFAR-10 数据集演示）
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

# 如果已经在 Google Drive 下载了，可以Drive 路径，这里默认下载到 Colab 本地
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=128, shuffle=True)

# 步骤 3: 搭建网络模型并转移到 GPU
class MyModel(nn.Module):
    def __init__(self):
        super(MyModel, self).__init__()
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(32 * 32 * 3, 512),
            nn.ReLU(),
            nn.Linear(512, 128),
            nn.ReLU(),
            nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.classifier(x)

# 实例化模型
model = MyModel()

# 核心点 1: 将网络模型转移到 GPU，也可以写成 model = model.cuda()
model = model.to(device)

# 步骤 4: 定义损失函数与优化器并转移到 GPU
# 核心点 2: 将损失函数转移到 GPU，也可以写成 loss_fn = nn.CrossEntropyLoss().cuda()
loss_fn = nn.CrossEntropyLoss().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# 步骤 5: 开始训练并记录时间
print("开始训练...")

# 对应知识点：记录开始时间
start_time = time.time()

# 为了演示时间对比，只跑 2 个 Epoch
epochs = 2

for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        # 获取输入数据（imgs 是输入，labels 是标注）
        imgs, labels = data

        # 核心点 3: 将数据和标注转移到 GPU，并且必须重新赋值。也可以写成：imgs = imgs.cuda(), labels = labels.cuda()
        imgs = imgs.to(device)
        labels = labels.to(device)

        # 梯度清零
        optimizer.zero_grad()

        # 前向传播
        outputs = model(imgs)

        # 计算损失
        loss = loss_fn(outputs, labels)

        # 反向传播
        loss.backward()

        # 优化权重
        optimizer.step()

        running_loss += loss.item()
        if i % 100 == 99:    # 每 100 个 batch 打印一次进度
            print(f'[Epoch {epoch + 1}, Batch {i + 1}] loss: {running_loss / 100:.3f}')
            running_loss = 0.0

# 对应知识点：记录结束时间并计算差值
end_time = time.time()
total_time = end_time - start_time

print("----------------------------------------")
print(f"训练结束！总耗时: {total_time:.2f} 秒")
print("----------------------------------------")


完整的训练流程和loss可视化

In [ ]:
import torch
import torchvision
from torch import nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
import os

# 1. 准备硬件设备（融合上节课的 GPU 知识，加速训练）
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前正在使用的训练设备是: {device}")

# 2. 准备数据集与数据加载器 (DataLoader)
# 下载并做基础预处理：转换为张量 (ToTensor)
train_data = torchvision.datasets.CIFAR10(root="../data", train=True,
                                          transform=torchvision.transforms.ToTensor(), download=True)
test_data = torchvision.datasets.CIFAR10(root="../data", train=False,
                                         transform=torchvision.transforms.ToTensor(), download=True)

# 打印数据集长度
train_data_size = len(train_data)
test_data_size = len(test_data)
print(f"训练数据集的长度为: {train_data_size}")  # 50000
print(f"测试数据集的长度为: {test_data_size}")   # 10000

# 利用 DataLoader 加载数据集
train_dataloader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
test_dataloader = DataLoader(test_data, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

# 3. 搭建神经网络模型 (Tudui 网络结构)
class Tudui(nn.Module):
    def __init__(self):
        super(Tudui, self).__init__()
        # 使用 nn.Sequential 容器将多个层组合在一起，精简代码
        self.model = nn.Sequential(
            # 这里的 padding=2 是小土堆为了保持尺寸或演示精心计算的
            nn.Conv2d(3, 32, 5, 1, 2),   # 输入3通道，输出32通道，卷积核5x5
            nn.MaxPool2d(2),             # 2x2最大池化
            nn.Conv2d(32, 32, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Flatten(),                # 展平层：将多维特征图展平为一维向量
            nn.Linear(64 * 4 * 4, 64),   # 全连接层
            nn.Linear(64, 10)            # 最终输出10个类别的得分
        )

    def forward(self, x):
        x = self.model(x)
        return x

# 实例化模型，并移至 GPU
tudui = Tudui().to(device)

# 4. 定义损失函数与优化器
loss_fn = nn.CrossEntropyLoss().to(device) # 分类任务标准损失函数，移至 GPU

learning_rate = 1e-2 # 即 0.01
optimizer = torch.optim.SGD(tudui.parameters(), lr=learning_rate) # 随机梯度下降优化器

# 5. 设置训练的一些关键控制参数与 TensorBoard 初始化
total_train_step = 0 # 记录总共训练了多少个 batch
total_test_step = 0  # 记录总共测试了多少轮
epoch = 25            # 训练的轮数（Colab 测试跑 5 轮即可看到明显趋势）

# 添加 tensorboard 可视化工具
writer = SummaryWriter("../logs_train")

# 6. 核心训练与测试大循环
for i in range(epoch):
    print(f"-------第 {i + 1} 轮训练开始-------")

    # 【训练步骤开始】
    tudui.train() # 规范写法：将模型设置为训练模式
    for data in train_dataloader:
        imgs, targets = data
        imgs = imgs.to(device)       # 数据移至 GPU
        targets = targets.to(device) # 标签移至 GPU

        # 优化器标准三步法
        optimizer.zero_grad() # 1. 梯度清零
        outputs = tudui(imgs) # 2. 前向传播得到输出
        loss = loss_fn(outputs, targets) # 3. 计算当前的损失

        loss.backward()       # 4. 反向传播，计算梯度
        optimizer.step()      # 5. 根据梯度更新模型权重

        total_train_step += 1

        # 每训练 100 次，打印一次日志并记录到 TensorBoard
        if total_train_step % 100 == 0:
            print(f"训练次数: {total_train_step}, Loss: {loss.item()}")
            # 将训练 Loss 写入 TensorBoard，纵轴是 loss 值，横轴是总训练步数
            writer.add_scalar("train_loss", loss.item(), total_train_step)

    # 测试步骤开始（收尾部分）
    tudui.eval() # 规范写法：将模型设置为评估模式
    total_test_loss = 0 # 每次测试前，清零测试集总 Loss

    # 测试保护：关闭梯度计算，省显存且加速
    with torch.no_grad():
        for data in test_dataloader:
            imgs, targets = data
            imgs = imgs.to(device)
            targets = targets.to(device)

            outputs = tudui(imgs)
            loss = loss_fn(outputs, targets)

            # 累加测试集上每个 batch 的 loss
            total_test_loss += loss.item()

    print(f"整体测试集上的 Loss: {total_test_loss}")
    # 将测试集总 Loss 记录到 TensorBoard
    writer.add_scalar("test_loss", total_test_loss, total_test_step)
    total_test_step += 1 # 测试轮数递增

    # 保存每一轮训练完的模型权重
    # 如果在 Colab 中，通常会保存在当前的运行目录下
    model_save_path = f"tudui_{i + 1}.pth"
    torch.save(tudui.state_dict(), model_save_path)
    print(f"模型已保存至: {model_save_path}")

# 关闭 SummaryWriter，否则数据可能无法正确写入文件
writer.close()
print("训练全部结束，TensorBoard 日志已关闭。")


In [ ]:
!kill 4293

In [ ]:
# 注意：重复加载时使用 %reload_ext，而不是 %load_ext
%reload_ext tensorboard
%tensorboard --logdir=../logs_train


下载一张图看它会分类为啥

In [ ]:
import torch
import torchvision.transforms as transforms
from PIL import Image
import requests
from io import BytesIO

# 步骤 1: 从网络获取一张非常清晰的猫咪正面图片链接
img_url = "https://images.unsplash.com/photo-1519052537078-e6302a4968d4?q=80&w=256&auto=format&fit=crop"

print("正在从链接下载图片...")
response = requests.get(img_url)
# 使用 PIL 库打开并转换为 RGB 格式
image = Image.open(BytesIO(response.content)).convert("RGB")

# 在 Colab 里展示一下这张猫咪图片
display(image)

# 步骤 2: 数据预处理（必须和训练时的缩放、归一化完全一致）
# CIFAR-10 的图片尺寸是 32x32，所以我们要把下载的大图缩放到 32x32
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor() # 转换为 Tensor 并且像素值归一化到 [0, 1]
])

img_tensor = transform(image)
print(f"处理后的单张图片形状: {img_tensor.shape}") # 此时是torch.Size([3, 32, 32])

# 关键：增加一个维度代表 batch_size = 1
img_tensor = img_tensor.unsqueeze(0)
print(f"增加批次维度后的形状: {img_tensor.shape}") # 变为 torch.Size([1, 3, 32, 32])

# 步骤 3: 载入刚训练好的模型并进行预测
# 检测设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. 实例化一个干净的网络结构（用Tudui 网络）
model = Tudui().to(device)

# 2. 加载刚才保存的权重文件（假设加载第10轮训练完的权重）
# 如果模型还在内存里没断开，其实直接用 model = tudui 也行
model.load_state_dict(torch.load("tudui_10.pth", map_location=device))

# 3. 将模型设置为评估推理模式
model.eval()

# 步骤 4: 前向传播与类别解析
# CIFAR-10 数据集对应的 10 个类别标签（顺序不能错）
classes = ['飞机', '汽车', '鸟', '猫', '鹿', '狗', '青蛙', '马', '船', '卡车']

# 将图片数据送入 GPU
img_tensor = img_tensor.to(device)

# 关闭梯度计算 进行推理
with torch.no_grad():
    outputs = model(img_tensor)

print("\n模型输出的原始得分（Raw Outputs）:", outputs)

# argmax横向求最大值索引获取预测结果
predict_idx = outputs.argmax(1).item()

# 输出识别结果
print(f"emmm,我猜这张图片是: 【{classes[predict_idx]}】")

批量处理四张图片

In [ ]:
import torch
import torchvision.transforms as transforms
from torch import nn
from PIL import Image
import requests
from io import BytesIO
import matplotlib.pyplot as plt

# 1. 重新声明网络结构（防止内存清空后找不到结构）
class Tudui(nn.Module):
    def __init__(self):
        super(Tudui, self).__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 32, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 32, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5, 1, 2),
            nn.MaxPool2d(2),
            nn.Flatten(),
            nn.Linear(64 * 4 * 4, 64),
            nn.Linear(64, 10)
        )

    def forward(self, x):
        return self.model(x)

# 2. 硬件检测与模型权重复活
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 实例化一个新的网络，并移至对应的设备上
model = Tudui().to(device)

# 核心复：从保存在硬盘里的pth 文件直接把权重进去
# 注意：看日志，最新保存的应该是 tudui_10.pth，这里改成 10
model.load_state_dict(torch.load("tudui_10.pth", map_location=device))
model.eval() # 切换为评估推理模式

# 3. 准备 4 张不同图片的网络链接字典
image_urls = {
    "拉布拉多犬": "https://images.unsplash.com/photo-1543466835-0994f318991b?q=80&w=256&auto=format&fit=crop",
    "红色跑车": "https://images.unsplash.com/photo-1549727409-fa786968032c?q=80&w=256&auto=format&fit=crop",
    "翱翔的海鸥": "https://images.unsplash.com/photo-1518002170369-026048d08705?q=80&w=256&auto=format&fit=crop",
    "漂亮猫咪": "https://images.unsplash.com/photo-1519052537078-e6302a4968d4?q=80&w=256&auto=format&fit=crop"
}

classes = ['飞机', '汽车', '鸟', '猫', '鹿', '狗', '青蛙', '马', '船', '卡车']

# CIFAR-10 标准预处理
transform = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.ToTensor()
])

# 4. 批量循环处理、预测，并画出图像
print("正在从互联网抓取图片并进行推理展示...")

# 创建一个画板，一行容纳 4 张图，宽度 16，高度 4
plt.figure(figsize=(16, 4))

for idx, (name, url) in enumerate(image_urls.items()):
    try:
        # 下载图片
        response = requests.get(url, timeout=5)
        image = Image.open(BytesIO(response.content)).convert("RGB")

        # 预处理图片，并添加批次（batch）维度
        img_tensor = transform(image).unsqueeze(0).to(device)

        # 模型推理
        with torch.no_grad():
            outputs = model(img_tensor)

        # 获取预测结果
        predict_idx = outputs.argmax(1).item()
        pred_label = classes[predict_idx]

        # 用 Matplotlib 摆放位置（1行 4列，当前是第 idx+1 个位置）
        plt.subplot(1, 4, idx + 1)
        plt.imshow(image) # 显示原始下载的图片

        # 在图片的上方或下方显示预测结果
        # 如果预测对了解析为猫，标题显示为蓝色；如果错误了，显示为红色
        title_color = "blue" if pred_label == name.split(' ')[-1] else "red" # Simplified color logic
        plt.title(f"【{name}】\n模型预测: {pred_label}", fontsize=12, color=title_color)
        plt.axis('off') # 把坐标轴的刻度线关掉，只看纯图

    except Exception as e:
        print(f"图片【{name}】加载失败: {e}")

# 调整布局防止字叠在一起，并展示画布
plt.tight_layout()
plt.show()
print("\n全部识别完毕，展示～")